# Soluciones — Clase 22: Clustering y segmentación

> **Nota para el docente:** Este notebook contiene soluciones completas y comentadas. No lo compartas con los alumnos antes de que entreguen su tarea. Úsalo como guía de corrección y para las discusiones en clase.

In [ ]:
# Solución completa: importar todas las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

plt.rcParams['figure.figsize'] = (9, 5)
print('Librerías cargadas ✅')

## Solución Ejercicio 1 — Carga y exploración

In [ ]:
# Cargar dataset y explorar
# Si el dataset no existe, generamos datos de ejemplo para la demostración
import os

if os.path.exists('datasets/ventas_tienda.csv'):
    df = pd.read_csv('datasets/ventas_tienda.csv')
    print('Dataset real cargado.')
else:
    # Generar datos simulados representativos
    np.random.seed(42)
    n = 200
    # Tres grupos: volumen alto/bajo, precio alto/bajo
    g1 = pd.DataFrame({'cantidad_vendida': np.random.normal(300, 30, n//3),
                        'precio_unitario': np.random.normal(20, 3, n//3),
                        'margen_ganancia': np.random.normal(5, 1, n//3)})
    g2 = pd.DataFrame({'cantidad_vendida': np.random.normal(50, 10, n//3),
                        'precio_unitario': np.random.normal(150, 20, n//3),
                        'margen_ganancia': np.random.normal(40, 5, n//3)})
    g3 = pd.DataFrame({'cantidad_vendida': np.random.normal(120, 20, n//3),
                        'precio_unitario': np.random.normal(60, 10, n//3),
                        'margen_ganancia': np.random.normal(15, 3, n//3)})
    df = pd.concat([g1, g2, g3], ignore_index=True)
    print('Datos simulados generados (3 grupos reales).')

print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.describe().round(2)

## Solución Ejercicio 2 — Preparar y escalar

In [ ]:
# Seleccionar columnas numéricas relevantes
# Elegimos las que tienen mayor varianza y sentido de negocio
cols_num = df.select_dtypes(include='number').columns.tolist()
col1, col2 = cols_num[0], cols_num[1]

X = df[[col1, col2]].dropna()

# Escalar: es OBLIGATORIO antes de K-Means
# StandardScaler lleva cada columna a media=0 y desv=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Columnas elegidas: {col1}, {col2}')
print(f'Forma del array escalado: {X_scaled.shape}')
print(f'Media columna 1 (escalada): {X_scaled[:,0].mean():.6f}  ← debe ser ~0')
print(f'Desv. estándar columna 1 (escalada): {X_scaled[:,0].std():.6f}  ← debe ser ~1')

## Solución Ejercicio 4 — Método del Codo

In [ ]:
# Método del Codo: probar K de 1 a 10 y graficar la inercia
inercias = []
valores_k = range(1, 11)

for k in valores_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)
    print(f'K={k:2d} → Inercia: {km.inertia_:,.1f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(valores_k, inercias, marker='o', color='steelblue', linewidth=2, markersize=7)
# Marcar visualmente el codo sugerido (K=3 en datos simulados)
ax.axvline(x=3, color='red', linestyle='--', alpha=0.6, label='Codo sugerido (K=3)')
ax.set_xlabel('Número de clusters (K)')
ax.set_ylabel('Inercia')
ax.set_title('Método del Codo')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\n► En datos con 3 grupos bien definidos, el codo suele aparecer en K=3.')

## Solución Ejercicio 5 — Silhouette Score

In [ ]:
# Silhouette Score para K de 2 a 8
# El mejor K tiene el score más alto
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    print(f'K={k}: {scores[k]:.3f}', '✅ MEJOR' if k == max(scores, key=scores.get) else '')

mejor_k = max(scores, key=scores.get)

plt.figure(figsize=(8, 4))
barras = plt.bar(list(scores.keys()), list(scores.values()),
                 color=['coral' if k != mejor_k else 'green' for k in scores])
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score por K (verde = mejor)')
plt.axhline(y=0.5, color='navy', linestyle='--', alpha=0.5, label='Umbral 0.5')
plt.legend()
plt.show()

print(f'\n► El mejor K según Silhouette es K={mejor_k} (score={scores[mejor_k]:.3f})')

## Solución Ejercicio 6 — Perfil de clusters e interpretación

In [ ]:
# Entrenar el modelo final con K=3
K_FINAL = 3
modelo_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
modelo_final.fit(X_scaled)

X_res = X.copy()
X_res['cluster'] = modelo_final.labels_

# Perfil promedio por cluster
perfil = X_res.groupby('cluster').agg(['mean', 'count']).round(2)
print('Perfil de clusters:')
print(perfil)

# Interpretar los clusters con nombres de negocio
print('\n► Interpretación:')
promedios = X_res.groupby('cluster')[col1].mean()
orden = promedios.sort_values().index.tolist()
nombres = ['Volumen bajo / precio alto (Premium)',
           'Volumen medio (Estándar)',
           'Volumen alto / precio bajo (Masivo)']
for rank, (cluster_id, nombre) in enumerate(zip(orden, nombres)):
    n = (X_res['cluster'] == cluster_id).sum()
    print(f'  Cluster {cluster_id}: {nombre} — {n} registros')

In [ ]:
# Visualización final con colores claros y centroides marcados
centros = scaler.inverse_transform(modelo_final.cluster_centers_)
paleta = ['#2196F3', '#FF5722', '#4CAF50']

plt.figure(figsize=(9, 6))
for i in range(K_FINAL):
    mask = X_res['cluster'] == i
    plt.scatter(
        X_res.loc[mask, col1], X_res.loc[mask, col2],
        c=paleta[i], label=f'Cluster {i}',
        alpha=0.7, edgecolors='k', linewidths=0.3, s=60
    )

plt.scatter(centros[:, 0], centros[:, 1],
            marker='X', s=350, color='black',
            zorder=10, label='Centroides')

for i, centro in enumerate(centros):
    plt.annotate(f'  C{i}', (centro[0], centro[1]),
                 fontsize=11, fontweight='bold')

plt.xlabel(col1)
plt.ylabel(col2)
plt.title(f'Segmentación K-Means (K={K_FINAL}) — Ventas Tienda')
plt.legend()
plt.tight_layout()
plt.show()

score_final = silhouette_score(X_scaled, modelo_final.labels_)
print(f'Silhouette Score del modelo final: {score_final:.3f}')

## Solución — Dataset de estudiantes

In [ ]:
# Generar datos simulados de estudiantes si no existe el CSV
np.random.seed(0)
n_est = 180

alto = pd.DataFrame({
    'nota_matematicas': np.random.normal(88, 5, n_est//3),
    'nota_ciencias': np.random.normal(85, 6, n_est//3),
    'horas_estudio': np.random.normal(5, 0.5, n_est//3),
    'faltas': np.random.normal(1, 0.5, n_est//3)
})
bajo = pd.DataFrame({
    'nota_matematicas': np.random.normal(45, 7, n_est//3),
    'nota_ciencias': np.random.normal(50, 8, n_est//3),
    'horas_estudio': np.random.normal(1.5, 0.5, n_est//3),
    'faltas': np.random.normal(8, 2, n_est//3)
})
medio = pd.DataFrame({
    'nota_matematicas': np.random.normal(68, 5, n_est//3),
    'nota_ciencias': np.random.normal(65, 6, n_est//3),
    'horas_estudio': np.random.normal(3, 0.5, n_est//3),
    'faltas': np.random.normal(4, 1, n_est//3)
})

df_est = pd.concat([alto, bajo, medio], ignore_index=True)
cols_est = ['nota_matematicas', 'nota_ciencias', 'horas_estudio', 'faltas']
X_est = df_est[cols_est]

# Escalar y aplicar K-Means
scaler_est = StandardScaler()
X_est_scaled = scaler_est.fit_transform(X_est)

modelo_est = KMeans(n_clusters=3, random_state=42, n_init=10)
modelo_est.fit(X_est_scaled)
df_est['cluster'] = modelo_est.labels_

perfil_est = df_est.groupby('cluster')[cols_est].mean().round(2)
conteo_est = df_est['cluster'].value_counts().sort_index()

print('Perfil por cluster (estudiantes):')
print(perfil_est)
print(f'\nConteo: {conteo_est.to_dict()}')

# Los clusters deben corresponder a: alto rendimiento, bajo rendimiento, rendimiento medio
print('\n► Interpretación pedagógica:')
for i in range(3):
    notas = perfil_est.loc[i, 'nota_matematicas']
    if notas > 80:
        label = 'Alto rendimiento — estudia mucho, pocas faltas'
    elif notas < 60:
        label = 'Bajo rendimiento — pocas horas de estudio, muchas faltas'
    else:
        label = 'Rendimiento medio — puede mejorar con apoyo'
    print(f'  Cluster {i}: {label}')